# Native Sparse Attention, part 0: why long attention is slow
*Companion notebooks to Yuan et al., 2025, [Native Sparse Attention: Hardware-Aligned and Natively Trainable Sparse Attention](https://arxiv.org/abs/2502.11089) (DeepSeek-AI and Peking University). Part 0 of 7.*

**TL;DR.** A language model that reads a 64,000-token document lets every new token look back at every earlier token. That is accurate and expensive: the work grows with the *square* of the length, and during generation the cost is dominated by *copying* the memory of earlier tokens, not by arithmetic. Methods that skip most of the looking ("sparse attention") have existed for years, but Yuan et al. argue they rarely turn into real speedups and, because they are bolted on after training, can hurt quality. NSA (Native Sparse Attention) is their redesign. They report, on a 27B-parameter model (3B active) with 64k-token sequences on A100 GPUs, a **9.0x faster forward pass, 6.0x faster backward pass and 11.6x faster decoding than FlashAttention-2**, with an average benchmark score of 0.456 against 0.443 for full attention.

<img src="nsa_assets/fig1_overview_results.png" width="700"/>

*Figure 1 of the paper. Left: NSA matches or beats full attention on general benchmarks, LongBench and reasoning. Right: speedups over full attention at 64k tokens. Notice that the decode speedup (11.6x) is the largest: that is where the memory bottleneck we describe below bites hardest.*

This notebook does **not** build NSA yet. We first build the problem, so that each of NSA's design choices in parts 1-6 reads as an answer to something specific. Everything here runs on a CPU in a few seconds.

| part | what we build | the number it carries |
|---|---|---|
| **0 (this one)** | what attention costs and why sparsity alone disappointed | 70-80% of 64k decode latency is attention (cited by the paper) |
| 1 | token compression | l=32, d=16 gives about 4,094 compressed tokens at 64k |
| 2 | blockwise token selection | n=16 blocks of 64 gives 1,024 tokens |
| 3 | sliding window | w=512 tokens |
| 4 | gating and the full layer | about 5,632 tokens read per query at 64k |
| 5 | hardware alignment | 4x / 6.4x / 9.1x / 11.6x expected decode speedup |
| 6 | native trainability | what goes wrong when sparsity is added after training |

*A note on voice.* "We" in these notebooks means *you and us running the code*. Results from the paper are always written as "Yuan et al. report". Where we measure something ourselves, it is on a toy, and we say so.

In [ ]:
import math, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"   # used only where a notebook trains
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
CMP_C, SLC_C, WIN_C, FULL_C = "#1c7ed6", "#d9480f", "#2f9e44", "#495057"      # one colour per branch, used in every notebook
NEG = -1e30                                                                     # a finite "minus infinity": a fully masked row gives uniform weights, never NaN

CHECKS = []
def check(name, ok, detail=""):
    """Record + print PASS/FAIL. Every claim in these notebooks is backed by one of these lines."""
    CHECKS.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))

## Attention on four tokens

Before any theory, one concrete case. Take four tokens, each described by two numbers. In attention every token plays three roles: it asks a **query** ("what am I looking for?"), it advertises a **key** ("what do I contain?"), and it carries a **value** ("what do I hand over if you pick me?").

Token 3 (the last one) wants to build its output. It compares its query to every key with a dot product, turns the four scores into weights that sum to 1 (softmax), and returns the weighted average of the values. A token may only look at itself and earlier tokens ("causal"), because during generation the future does not exist yet.

In [ ]:
# Four tokens, two numbers each. We compute the output of the LAST token by hand, then compare with the library.
q = torch.tensor([1.0, 0.0])                                         # query of token 3: "I am looking for something like [1, 0]"
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5], [1.0, 0.0]])   # keys of tokens 0..3
V = torch.tensor([[10., 0.], [0., 10.], [5., 5.], [8., 2.]])         # values of tokens 0..3

scores = K @ q / math.sqrt(2)                                        # similarity of the query with each key
weights = scores.softmax(0)                                          # turn into probabilities
out = weights @ V                                                    # weighted average of the values
print("scores :", scores.numpy().round(3))
print("weights:", weights.numpy().round(3), " (sum =", round(weights.sum().item(), 3), ")")
print("output :", out.numpy().round(3))
ref = F.scaled_dot_product_attention(q[None, None, None], K[None, None], V[None, None])[0, 0, 0]
check("hand-computed attention equals PyTorch's scaled_dot_product_attention", torch.allclose(out, ref, atol=1e-6))
check("tokens 0 and 3 have the same key as the query, so they get the same (largest) weight", torch.isclose(weights[0], weights[3]) and weights.argmax() in (0, 3))

Tokens 0 and 3 have keys identical to the query, so they get the biggest and equal weights; token 1 (orthogonal) gets the smallest. The output sits closest to the values of tokens 0 and 3.

In general, for a query at position *t* (the paper's equations 1 and 2):

&nbsp;&nbsp;&nbsp;&nbsp;`o_t = sum_i  w(t,i) * v_i`, with `w(t,i) = softmax_i( q_t . k_i / sqrt(d_k) )`, over `i <= t`.

Here `d_k` is the key dimension; dividing by `sqrt(d_k)` stops the dot products from growing with it. We write this once, for the layout NSA uses: `G` key/value groups, each shared by `H` query heads (**GQA**, grouped-query attention; it shrinks the memory of earlier tokens, the "KV cache", by letting heads share keys and values).

In [ ]:
def full_causal_attention(q, k, v):
    """Ordinary causal attention, with the grouped-query layout NSA uses.
    q: (B, G, H, T, dk)   G key/value groups, H query heads share each group's K and V (GQA)
    k: (B, G, T, dk)      v: (B, G, T, dv)       returns (output (B,G,H,T,dv), weights (B,G,H,T,T))"""
    T, dk = q.shape[-2], q.shape[-1]
    scores = torch.einsum("bghtd,bgsd->bghts", q, k) / math.sqrt(dk)          # how much query t likes key s
    causal = torch.ones(T, T, dtype=torch.bool, device=q.device).tril()        # token t may only look at tokens <= t
    p = scores.masked_fill(~causal, NEG).softmax(-1)                           # each row sums to 1
    return torch.einsum("bghts,bgsd->bghtd", p, v), p

In [ ]:
B, G, H, T, dk = 2, 2, 4, 64, 16
q, k, v = torch.randn(B, G, H, T, dk), torch.randn(B, G, T, dk), torch.randn(B, G, T, dk)
out, p = full_causal_attention(q, k, v)
ref = F.scaled_dot_product_attention(q, k[:, :, None].expand(-1, -1, H, -1, -1), v[:, :, None].expand(-1, -1, H, -1, -1), is_causal=True)
check("our grouped-query causal attention equals PyTorch's (is_causal=True)", torch.allclose(out, ref, atol=1e-5))
check("every row of attention weights sums to 1", torch.allclose(p.sum(-1), torch.ones(B, G, H, T), atol=1e-5))
check("no token looks at the future (weights above the diagonal are 0)", p.triu(1).abs().max() == 0)

That is the whole baseline. The matrix `p` has one row per query and one column per key: **T x T numbers**. That square is the problem.

## Counting the work

For a sequence of T tokens, causal attention scores every (query, earlier key) pair: 1 + 2 + ... + T = T(T+1)/2 pairs per head. Each pair costs a dot product and, later, a weighted sum, so total arithmetic grows like T squared.

In [ ]:
Ts = np.array([1_000, 8_000, 16_000, 32_000, 64_000, 128_000])
pairs = Ts * (Ts + 1) / 2
flops_per_head = pairs * 2 * (192 + 128)                       # 2 FLOPs per multiply-add; QK^T uses d_k, AV uses d_v
print(f"{'tokens':>9s} {'pairs per head':>16s} {'GFLOPs per head per layer':>28s}")
for t_, pr, f in zip(Ts, pairs, flops_per_head): print(f"{t_:>9,d} {pr:>16,.0f} {f / 1e9:>28,.1f}")
# check the pair count against an actual mask
m = torch.ones(300, 300, dtype=torch.bool).tril()
check("the number of causal (query, key) pairs is T(T+1)/2", m.sum().item() == 300 * 301 // 2)
check("doubling the length multiplies the work by about 4 (quadratic)", 3.9 < pairs[3] / pairs[2] < 4.1, f"32k vs 16k: {pairs[3] / pairs[2]:.2f}x")
fig, ax = plt.subplots(figsize=(6.2, 3.3)); ax.loglog(Ts, pairs, "o-", color=FULL_C, lw=2); ax.set_xlabel("sequence length T"); ax.set_ylabel("causal (query, key) pairs per head")
ax.set_title("Full attention: pairs grow as T squared"); plt.tight_layout(); plt.show()

64,000 tokens means about 2 billion pairs per head per layer, and a model has dozens of layers and dozens of heads. Skipping most pairs is clearly attractive. But whether skipping translates into *time saved* depends on what the hardware is actually waiting for, which is the next idea.

## Two ways to be slow: arithmetic intensity

**Concrete picture.** A chef (the GPU's arithmetic units) can chop extremely fast. Ingredients come from a pantry down the hall (memory), carried at a fixed rate. If each ingredient needs one chop, the chef stands idle waiting for deliveries: the kitchen is *delivery-bound*. If each ingredient needs a thousand chops, the deliveries are never the problem: it is *chop-bound*.

**Definition.** *Arithmetic intensity* is floating-point operations (FLOPs) performed per byte moved from memory. Each GPU has a *critical* intensity, peak FLOPs divided by memory bandwidth. Above it a kernel is compute-bound, below it memory-bound. For an A100 (public datasheet figures: about 312 trillion bf16 FLOPs per second, about 2.0 TB/s of HBM, high-bandwidth memory, on the 80 GB card) the critical value is about 156 FLOPs per byte. We did not measure these.

Now the two phases of using a language model:

* **Training and prefill** (reading the prompt): all T queries are processed together. Tiled kernels such as FlashAttention keep the score matrix in fast on-chip memory (SRAM), so memory traffic is only the inputs and outputs, about linear in T, while the arithmetic is quadratic. Intensity grows with T.
* **Decoding** (generating one token at a time): one new query must read the entire KV cache, every key and value of every earlier token, and do two multiply-adds per element. Intensity is stuck near 1 FLOP per byte *no matter how long the context is*.

In [ ]:
A100_FLOPS, A100_BW = 312e12, 2.0e12                    # datasheet peak bf16 FLOP/s and HBM bytes/s (80 GB card)
crit = A100_FLOPS / A100_BW
dk, dv, bytes_per = 192, 128, 2                          # paper head sizes, bf16 = 2 bytes
def intensity_prefill(T):
    """Ideal tiled attention over T tokens, one head: causal FLOPs / (read Q,K,V once + write O once)."""
    flops = T * T * (dk + dv)                            # 2 * (T^2/2 pairs) * (dk + dv)
    traffic = T * (dk + dk + dv + dv) * bytes_per        # Q, K, V in; O out
    return flops / traffic
def intensity_decode(s):
    """One new query against s cached tokens, one head: 2 s (dk + dv) FLOPs / read the whole K and V cache."""
    return 2 * s * (dk + dv) / (s * (dk + dv) * bytes_per)
for T in (1_000, 8_000, 64_000):
    print(f"T = {T:>6,d}   prefill intensity {intensity_prefill(T):>8,.0f} FLOP/byte   decode intensity {intensity_decode(T):.1f} FLOP/byte")
print(f"A100 critical intensity = {crit:.0f} FLOP/byte  (above: compute-bound, below: memory-bound)")

xs = np.logspace(0, 4, 100); fig, ax = plt.subplots(figsize=(7, 3.6))
ax.loglog(xs, np.minimum(xs * A100_BW, A100_FLOPS) / 1e12, color=FULL_C, lw=2, label="A100 roofline")
for lab, x_, c in [("decode (any length)", intensity_decode(64_000), SLC_C), ("prefill/training T=8k", intensity_prefill(8_000), CMP_C)]:
    ax.plot(x_, min(x_ * A100_BW, A100_FLOPS) / 1e12, "o", ms=10, color=c, label=lab)
ax.set_xlabel("arithmetic intensity (FLOP per byte)"); ax.set_ylabel("attainable TFLOP/s"); ax.set_title("Where the two phases sit on the roofline"); ax.legend(); plt.tight_layout(); plt.show()
check("decoding sits far below the A100 critical intensity (memory-bound)", intensity_decode(64_000) < crit / 50, f"{intensity_decode(64_000):.1f} vs {crit:.0f}")
check("long-sequence prefill sits far above it (compute-bound)", intensity_prefill(8_000) > crit * 5, f"{intensity_prefill(8_000):.0f} vs {crit:.0f}")
check("decode intensity does not depend on context length", abs(intensity_decode(1_000) - intensity_decode(64_000)) < 1e-9)

This is the first design constraint of NSA, and it points in two different directions:

| phase | bottleneck | what must shrink |
|---|---|---|
| training, prefill | arithmetic (compute-bound) | the number of query-key pairs |
| decoding | memory traffic (memory-bound) | the number of KV-cache **bytes read** per new token |

A sparse method can win in one phase and do nothing in the other, or reduce arithmetic while leaving bytes unchanged. The next section is about how that happens in practice. For the decoding side, Yuan et al. cite estimates that softmax attention accounts for **70-80% of total decoding latency at 64k tokens**, which is why a method that shrinks bytes matters.

## Why earlier sparse methods under-delivered

Yuan et al. identify two problems with inference-time sparsity, both of which are about the table above.

**1. Phase-restricted sparsity.** Some methods (H2O) sparsify decoding but need expensive preprocessing in prefill; others (MInference) speed up prefill and leave decoding as expensive as before. A method that helps one phase leaves the other at full price, which matters for workloads dominated by that other phase: book summarisation is prefill-heavy, long chain-of-thought reasoning is decode-heavy.

**2. Per-head selection meets GQA.** With GQA, several query heads share one KV cache. If each head *independently* picks its own top-k blocks (as Quest does), the bytes that must be loaded are the **union** of all the heads' choices, not any single head's. Computation per head stays sparse; memory traffic does not. A concrete number, using the paper's shape of 16 query heads per KV group and a 64k context cut into 1,024 blocks of 64 tokens, each head choosing 16 blocks:

In [ ]:
nb_blocks, n_pick, heads = 1024, 16, 16                    # 64k tokens / 64-token blocks; 16 blocks per head; 16 heads share one KV group
rng = np.random.default_rng(0)
union_sizes = [len(set(np.concatenate([rng.choice(nb_blocks, n_pick, replace=False) for _ in range(heads)]))) for _ in range(2000)]
sim = np.mean(union_sizes) / nb_blocks
formula = 1 - (1 - n_pick / nb_blocks) ** heads            # P(a given block is chosen by at least one of the heads)
print(f"each head reads {n_pick / nb_blocks:.1%} of the blocks (compute sparsity: {nb_blocks / n_pick:.0f}x)")
print(f"if heads choose independently, the GROUP must load {sim:.1%} of the blocks  ->  memory sparsity only {1 / sim:.1f}x")
check("simulation matches the closed form 1 - (1 - n/N)^heads", abs(sim - formula) < 0.005, f"sim {sim:.4f} vs formula {formula:.4f}")
check("independent per-head selection loses most of the memory saving (union > 10x one head's share)", sim > 10 * n_pick / nb_blocks)

hs = np.arange(1, 17); fig, ax = plt.subplots(figsize=(6.4, 3.3))
ax.plot(hs, 100 * (1 - (1 - n_pick / nb_blocks) ** hs), "o-", color=SLC_C, lw=2, label="heads choose independently")
ax.plot(hs, np.full_like(hs, 100 * n_pick / nb_blocks, dtype=float), "s-", color=WIN_C, lw=2, label="heads share one choice (NSA)")
ax.set_xlabel("query heads sharing a KV group"); ax.set_ylabel("% of the KV cache that must be loaded"); ax.set_title("Per-head selection vs. group-consistent selection"); ax.legend(); plt.tight_layout(); plt.show()

**Cost of this number, stated plainly.** This is the *worst case* in which heads choose uncorrelated blocks. Real heads in a trained model partially agree, so the true union lies between the two lines; we did not measure it on a real model. What it shows is the mechanism: NSA's response (part 2) is to *force* the heads of a group to agree by summing their block scores (the paper's equation 10), so the group loads the single-head amount.

## Why training is the other half of the story

Yuan et al. also argue that most sparse methods only touch inference: a model is trained with full attention and sparsified afterwards. Two consequences, both of which we will test ourselves in part 6:

* **Quality.** The trained model's attention is not sparse. The paper cites Chen et al., 2024, for the finding that the top 20% of attention weights cover only about 70% of the total attention mass, so cutting after the fact discards a lot.
* **Training cost.** Many selection tricks (k-means clustering, hashing, sorting) are *discrete*: they produce integer indices, and gradients cannot flow through an integer. A small demonstration of what that means:

In [ ]:
scores = torch.randn(10, requires_grad=True)                 # "how important is block i", as some network would produce
values = torch.arange(10.0) + 1                             # something downstream that depends on WHICH blocks are chosen

idx = scores.topk(3).indices                                # hard top-3 selection: the output is a list of INTEGERS
loss_hard = values[idx].sum()
print("hard top-k : loss.requires_grad =", loss_hard.requires_grad, "  (the graph is cut at the integer indices)")

loss_soft = (scores.softmax(0) * values).sum()              # a soft, weighted version of the same idea
loss_soft.backward()
print("soft weights: gradient reaching the scores =", scores.grad.numpy().round(2))
check("hard top-k gives the scores NO gradient: the loss is disconnected from them", not loss_hard.requires_grad)
check("a soft (differentiable) path does send gradient to the scores", bool(scores.grad.abs().sum() > 0))

The loss clearly depends on *which* blocks were chosen, yet the scores that made the choice are cut out of the computation graph: the model cannot learn to select better from this signal. NSA's answer, previewed here and built in part 2, is to derive the selection scores from a branch that *is* differentiable (the compressed attention), so the scores improve as a side effect of training, with no extra loss.

## NSA in one picture

Instead of attending to all `t` earlier tokens, each query attends to a **small set built from three sources**, and mixes the three answers with learned gates:

<img src="nsa_assets/fig2_architecture.png" width="760"/>

*Figure 2 of the paper. Left: the three parallel branches for one query. Right: which (query, key) pairs each branch computes (green) in the attention matrix. Notice the compressed branch covers the whole past coarsely, the selected branch covers a few chosen regions precisely, and the window covers only the nearest tokens.*

| branch | covers | resolution | part |
|---|---|---|---|
| compressed | the whole past | one vector per block of tokens | 1 |
| selected | the most relevant blocks | every token, but only in chosen blocks | 2 |
| sliding window | the last `w` tokens | every token | 3 |

With Yuan et al.'s configuration (compression block `l=32`, stride `d=16`, selection block `l'=64`, `n=16` selected blocks, window `w=512`), how many tokens does one query touch at 64k context?

In [ ]:
l, d, lp, n, w = 32, 16, 64, 16, 512                         # paper configuration
print(f"{'context':>9s} {'compressed':>11s} {'selected':>9s} {'window':>7s} {'total':>7s} {'full attn':>10s} {'ratio':>7s}")
for s in (8192, 16384, 32768, 65536):
    cmp_ = (s - l) // d + 1; total = cmp_ + n * lp + w
    print(f"{s:>9,d} {cmp_:>11,d} {n * lp:>9,d} {w:>7,d} {total:>7,d} {s:>10,d} {s / total:>6.1f}x")
exact = (65536 - l) // d + 1 + n * lp + w
paper_table4 = 65536 / 5632
check("at 64k an NSA query reads about 5.6k tokens (paper's Table 4 lists 5,632)", abs(exact - 5632) <= 4, f"we count {exact}")
check("that is an 11.6x reduction in tokens read, the paper's expected decode speedup", abs(paper_table4 - 11.6) < 0.05, f"65536/5632 = {paper_table4:.2f}")

So the *expected* decode speedup of 11.6x in the paper is simply the ratio of tokens read, 65,536 over 5,632. Because decoding is memory-bound, time follows bytes. (Our own count (5,631) differs from the paper's 5,632 by one token; the paper's number equals `s/d + n*l' + w` while the exact block count is `(s-l)/d + 1`. Immaterial, but we mention it so you are not surprised.)

## Closing thoughts

Sparse attention is an old idea; NSA's contribution, as Yuan et al. frame it, is that sparsity must be designed for the hardware's bottleneck in *both* phases and must exist during *training*. The remaining notebooks build each piece: compression (part 1), selection (part 2), the window (part 3), the gated layer (part 4), the hardware argument (part 5), and the training argument (part 6).

**What we did not do here:** time anything on a GPU, or check any number that depends on a trained 27B model. Every paper figure quoted above is Yuan et al.'s, not ours.

**Claims that lack a number or baseline in this notebook** (so you can decide whether to fix them): (1) "memory-bound decoding" rests on A100 datasheet figures, not on measurement; (2) the group-union result is a worst case for independent heads, not a measurement on a trained model; (3) the 70-80% attention share of decode latency and the 70%-of-mass figure are cited from the paper, not verified by us.

In [ ]:
print(f"{sum(ok for _, ok in CHECKS)}/{len(CHECKS)} checks passed")
for n, ok in CHECKS:
    if not ok: print("FAILED:", n)
assert all(ok for _, ok in CHECKS), "some checks failed"